# Tree Cover Change Analysis in Protected Areas
### CoRE Stack Innovation Challenge Demo - February 2026

This notebook demonstrates tree cover change analysis within protected areas using:
- **Tree cover change layers** from CoRE stack GEE app
- **Protected Areas Database** for India (IIT Bombay)
- **CoRE Stack API** for regional data access

---

## 1. Setup and Imports

In [2]:
import ee
import geemap
import requests
import json
from IPython.display import display, HTML
from google.colab import userdata

# Initialize Earth Engine
try:
    ee.Initialize(project='invertible-fin-447406-g5')
    print("✓ Earth Engine initialized successfully")
except:
    ee.Authenticate()
    ee.Initialize(project='invertible-fin-447406-g5')
    print("✓ Earth Engine authenticated and initialized")

✓ Earth Engine authenticated and initialized


## 2. Configuration

Set your location parameters and API credentials here:

In [3]:
# Location parameters
STATE = "odisha"
DISTRICT = "anugul"
TEHSIL = "anugul"

# API Configuration
API_KEY = userdata.get('API_key')  # Replace with your actual API key
API_URL = "https://geoserver.core-stack.org/api/v1/get_generated_layer_urls/"

# Protected Areas Asset
PROTECTED_AREAS_ASSET = "projects/ee-aaditeshwar/assets/protected-areas"

print(f"Configuration set for: {STATE.title()} > {DISTRICT.title()} > {TEHSIL.title()}")

Configuration set for: Odisha > Anugul > Anugul


## 3. Fetch Layer Information from CoRE Stack API

In [4]:
def get_layer_urls(state, district, tehsil, api_key):
    """
    Fetch layer URLs from CoRE Stack API
    """
    params = {
        'state': state.lower(),
        'district': district.lower(),
        'tehsil': tehsil.lower()
    }

    headers = {
        'X-API-Key': f'{api_key}'
    }

    try:
        response = requests.get(API_URL, params=params, headers=headers)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data: {e}")
        return None

# Fetch layers
layers_data = get_layer_urls(STATE, DISTRICT, TEHSIL, API_KEY)

if layers_data:
    print(f"✓ Successfully fetched {len(layers_data)} layers")
    for layer in layers_data:
        print(f"  - {layer['layer_name']} ({layer['layer_type']})")
else:
    print("✗ Failed to fetch layer data")

✓ Successfully fetched 85 layers
  - overall_change_vector_anugul_anugul (vector)
  - anugul_anugul_clart (raster)
  - ccd_raster_anugul_anugul_2022 (raster)
  - ccd_raster_anugul_anugul_2018 (raster)
  - ccd_raster_anugul_anugul_2019 (raster)
  - ccd_raster_anugul_anugul_2020 (raster)
  - ccd_raster_anugul_anugul_2021 (raster)
  - ccd_raster_anugul_anugul_2023 (raster)
  - ch_raster_anugul_anugul_2018 (raster)
  - ch_raster_anugul_anugul_2019 (raster)
  - ch_raster_anugul_anugul_2020 (raster)
  - ch_raster_anugul_anugul_2021 (raster)
  - ch_raster_anugul_anugul_2022 (raster)
  - ch_raster_anugul_anugul_2023 (raster)
  - ch_vector_anugul_anugul_2017_2023 (vector)
  - anugul_anugul (vector)
  - anugul_anugul_slope_percentage_raster (raster)
  - anugul_anugul_drought (vector)
  - restoration_anugul_anugul_vector (vector)
  - change_anugul_anugul_CropIntensity (raster)
  - LULC_21_22_anugul_anugul_level_2 (raster)
  - restoration_anugul_anugul_raster (raster)
  - change_anugul_anugul_Urba

## 4. Extract and Load GEE Assets

In [5]:
def extract_layer_by_name(layers, layer_name):
    """
    Extract specific layer from API response
    """
    for layer in layers:
        if layer['layer_name'] == layer_name:
            return layer
    return None

# Extract layers
hydrology_layer = extract_layer_by_name(layers_data, "deltaG_well_depth_anugul_anugul")
change_detection_layer = extract_layer_by_name(layers_data, "overall_change_raster_anugul_anugul")
lulc_layer = extract_layer_by_name(layers_data, "LULC_24_25_anugul_anugul_level_3")

# Load GEE assets
if hydrology_layer:
    tehsil_boundary = ee.FeatureCollection(hydrology_layer['gee_asset_path'])
    print(f"✓ Loaded Tehsil Boundary (Hydrology): {hydrology_layer['gee_asset_path']}")
else:
    print("✗ Hydrology layer not found")

if change_detection_layer:
    tree_change = ee.Image(change_detection_layer['gee_asset_path'])
    print(f"✓ Loaded Tree Change Detection: {change_detection_layer['gee_asset_path']}")
else:
    print("✗ Change Detection layer not found")

if lulc_layer:
    lulc = ee.Image(lulc_layer['gee_asset_path'])
    print(f"✓ Loaded LULC Level 3: {lulc_layer['gee_asset_path']}")
else:
    print("✗ LULC layer not found")

✓ Loaded Tehsil Boundary (Hydrology): projects/ee-corestackdev/assets/apps/mws/odisha/anugul/anugul/deltaG_well_depth_anugul_anugul
✓ Loaded Tree Change Detection: projects/ee-corestackdev/assets/apps/mws/odisha/anugul/anugul/overall_change_raster_anugul_anugul
✓ Loaded LULC Level 3: projects/ee-corestackdev/assets/apps/mws/odisha/anugul/anugul/anugul_anugul_2024-07-01_2025-06-30_LULCmap_10m


## 5. Load and Filter Protected Areas

Load protected areas database and filter for areas within the selected tehsil boundary.

In [6]:
# Load protected areas
protected_areas_all = ee.FeatureCollection(PROTECTED_AREAS_ASSET)

# Get tehsil boundary geometry
tehsil_geom = tehsil_boundary.geometry()

# Filter protected areas that intersect with tehsil boundary
protected_areas_filtered = protected_areas_all.filterBounds(tehsil_geom)

# Get count
pa_count = protected_areas_filtered.size().getInfo()

print(f"✓ Found {pa_count} protected area(s) in {TEHSIL.title()} tehsil")

if pa_count > 0:
    # Get names of protected areas
    pa_list = protected_areas_filtered.aggregate_array('name').getInfo()
    print("\nProtected Areas:")
    for i, name in enumerate(pa_list, 1):
        print(f"  {i}. {name}")
else:
    print("\nNote: No protected areas found in this tehsil. Visualizations will show tehsil boundary only.")

✓ Found 1 protected area(s) in Anugul tehsil

Protected Areas:
  1. Satkosia Gorge WLS


## 6. Define Visualization Parameters

In [7]:
# Tree cover change classes and colors
CHANGE_VIS_PARAMS = {
    'min': -2,
    'max': 5,
    'palette': ['FF0000', 'FFA500', 'FFFFFF', '8AFF8A', '007500', 'DEE64C', 'DEE64C', '000000']
}

CHANGE_LABELS = {
    -2: 'Tree Cover Loss',
    -1: 'Degradation',
    0: 'No Change',
    1: 'Improvement',
    2: 'Tree Cover Gain',
    3: 'Partially Degraded',
    4: 'Partially Degraded',
    5: 'Missing Data'
}

# LULC visualization (example - adjust as needed)
LULC_VIS_PARAMS = {
    'min': 0,
    'max': 15,
    'palette': [
        '#419BDF', '#397D49', '#88B053', '#7A87C6', '#E49635',
        '#DFC35A', '#C4281B', '#A59B8F', '#B39FE1', '#FF0000',
        '#686868', '#F5F5F5', '#CCCCCC', '#0000FF', '#8B4513', '#000000'
    ]
}

print("✓ Visualization parameters defined")

✓ Visualization parameters defined


## 7. Map 1: Tree Cover Change in Protected Areas

This map shows tree cover changes (2017-2023) overlaid on protected area boundaries within the selected tehsil.

In [8]:
# Create map centered on tehsil
center = tehsil_geom.centroid().coordinates().getInfo()
Map1 = geemap.Map(center=[center[1], center[0]], zoom=10)

# Add tehsil boundary
Map1.addLayer(
    tehsil_boundary,
    {'color': 'blue'},
    f'{TEHSIL.title()} Tehsil Boundary',
    opacity=0.5
)

# Clip tree change to tehsil boundary
tree_change_clipped = tree_change.clip(tehsil_geom)

# If protected areas exist, clip to them as well
if pa_count > 0:
    pa_geom = protected_areas_filtered.geometry()
    tree_change_pa = tree_change_clipped.clip(pa_geom)

    # Add tree cover change layer
    Map1.addLayer(
        tree_change_pa,
        CHANGE_VIS_PARAMS,
        'Tree Cover Change (2017-2023)',
        opacity=0.8
    )

    # Add protected area boundaries
    Map1.addLayer(
        protected_areas_filtered,
        {'color': 'black'},
        'Protected Area Boundaries',
        opacity=1
    )
else:
    # Show tree change for entire tehsil
    Map1.addLayer(
        tree_change_clipped,
        CHANGE_VIS_PARAMS,
        'Tree Cover Change (2017-2023)',
        opacity=0.8
    )

# Add legend
legend_dict = {}
for value, label in CHANGE_LABELS.items():
    color = CHANGE_VIS_PARAMS['palette'][value + 2]  # offset by 2 since min is -2
    legend_dict[label] = color

Map1.add_legend(title='Tree Cover Change', legend_dict=legend_dict, position='bottomright')

# Display map
Map1

Map(center=[20.842213055841757, 84.94014641057143], controls=(WidgetControl(options=['position', 'transparent_…

## 8. Pan India Protected Areas & Terrain

This section loads the **Pan India Protected Areas** dataset and the **Pan India Terrain Raster** (TPI-based landform classification) from the CoRE Stack GEE assets. Both are visualised together on an interactive map.

**Terrain classes (TPI-based):**
| Value | Class |
|-------|-------|
| 2 | Incised drainages and low ridges |
| 3 | Mountain tops and high ridges |
| 4 | U-shape valleys |
| 5 | Broad Flat Areas |
| 6 | Broad open slopes |
| 7 | Flat tops |
| 8 | Upper Slopes |
| 9 | Deep valleys and canyons |
| 10 | Incised drainages and low ridges |
| 11 | Mountain tops and high ridges |


In [9]:
# ── Assets ────────────────────────────────────────────────────────────────────
PAN_INDIA_PA_ASSET      = "projects/ee-aaditeshwar/assets/protected-areas"
PAN_INDIA_TERRAIN_ASSET = "projects/corestack-datasets/assets/datasets/terrain/pan_india_terrain_raster"

# Load assets
pan_india_pa      = ee.FeatureCollection(PAN_INDIA_PA_ASSET)
pan_india_terrain = ee.Image(PAN_INDIA_TERRAIN_ASSET)

print("✓ Loaded Pan India Protected Areas")
print("✓ Loaded Pan India Terrain Raster")

# ── Terrain vis params ────────────────────────────────────────────────────────
TERRAIN_VIS_PARAMS = {
    'min': 1,
    'max': 12,
    'palette': [
        '#313695',  # 1  – Deep valleys and canyons       (opacity 0 – transparent)
        '#4575b4',  # 2  – Incised drainages and low ridges
        '#a50026',  # 3  – Mountain tops and high ridges
        '#e0f3f8',  # 4  – U-shape valleys
        '#fffc00',  # 5  – Broad Flat Areas
        '#feb24c',  # 6  – Broad open slopes
        '#f46d43',  # 7  – Flat tops
        '#d73027',  # 8  – Upper Slopes
        '#313695',  # 9  – Deep valleys and canyons
        '#4575b4',  # 10 – Incised drainages and low ridges
        '#a50026',  # 11 – Mountain tops and high ridges
        '#ffffff',  # 12 – Background                    (opacity 0 – transparent)
    ]
}

TERRAIN_LEGEND = {
    'Incised drainages and low ridges': '4575b4',
    'Mountain tops and high ridges':    'a50026',
    'U-shape valleys':                  'e0f3f8',
    'Broad Flat Areas':                 'fffc00',
    'Broad open slopes':                'feb24c',
    'Flat tops':                        'f46d43',
    'Upper Slopes':                     'd73027',
    'Deep valleys and canyons':         '313695',
}

# ── Build map ─────────────────────────────────────────────────────────────────
from ipywidgets import HTML, VBox
import ipywidgets as widgets

Map3 = geemap.Map(center=[22.5, 82.0], zoom=5)

# 1. Terrain raster (bottom layer)
Map3.addLayer(
    pan_india_terrain,
    TERRAIN_VIS_PARAMS,
    'Pan India Terrain (TPI)',
    opacity=0.7
)

# 2. Protected area boundaries – styled with fill so clicks register easily
pa_styled = pan_india_pa.style(
    color='000000',
    fillColor='00000000',  # transparent fill, but geometry is still clickable
    width=1
)
Map3.addLayer(pa_styled, {}, 'Pan India Protected Areas')

# Terrain legend
Map3.add_legend(
    title='Terrain Classes (TPI)',
    legend_dict=TERRAIN_LEGEND,
    position='bottomright'
)

# ── Info panel widget ─────────────────────────────────────────────────────────
info_html = HTML(
    value="""
    <div style='
        background: white;
        border: 1px solid #ccc;
        border-radius: 6px;
        padding: 12px 16px;
        font-family: sans-serif;
        font-size: 13px;
        min-width: 260px;
        max-width: 340px;
        box-shadow: 2px 2px 6px rgba(0,0,0,0.15);
    '>
        <b style='font-size:14px;'>Protected Area Info</b><br>
        <hr style='margin:6px 0;border:none;border-top:1px solid #eee;'>
        <span style='color:#888;'>Click on a protected area boundary to see its metadata.</span>
    </div>
    """
)

def build_info_html(props):
    """Render a dict of PA properties as a styled HTML card."""
    # Fields to skip (geometry, internal IDs)
    skip = {'system:index', '.geo'}

    rows = ""
    for k, v in sorted(props.items()):
        if k in skip or v is None or v == '':
            continue
        rows += f"""
        <tr>
          <td style='color:#555;padding:3px 8px 3px 0;vertical-align:top;white-space:nowrap;'>
            <b>{k}</b>
          </td>
          <td style='padding:3px 0;word-break:break-word;'>{v}</td>
        </tr>"""

    if not rows:
        rows = "<tr><td colspan='2' style='color:#888;'>No properties available.</td></tr>"

    name = props.get('name', props.get('Name', props.get('PA_NAME', 'Protected Area')))

    return f"""
    <div style='
        background: white;
        color:#333;
        border: 1px solid #ccc;
        border-radius: 6px;
        padding: 12px 16px;
        font-family: sans-serif;
        font-size: 13px;
        min-width: 260px;
        max-width: 340px;
        box-shadow: 2px 2px 6px rgba(0,0,0,0.15);
    '>
        <b style='font-size:14px;color:#1a6b3a;'>🌿 {name}</b>
        <hr style='margin:6px 0;border:none;border-top:1px solid #eee;'>
        <table style='width:100%;border-collapse:collapse;'>
          {rows}
        </table>
    </div>
    """

# ── Click handler ─────────────────────────────────────────────────────────────
def handle_click(**kwargs):
    if kwargs.get('type') != 'click':
        return

    coords = kwargs.get('coordinates')   # [lat, lng]
    if not coords:
        return

    lat, lng = coords[0], coords[1]
    point = ee.Geometry.Point([lng, lat])

    # Update panel to show loading state
    info_html.value = """
    <div style='padding:12px 16px;font-family:sans-serif;font-size:13px;
                background:white;border:1px solid #ccc;border-radius:6px;
                box-shadow:2px 2px 6px rgba(0,0,0,0.15);'>
        <span style='color:#888;'>⏳ Querying...</span>
    </div>"""

    try:
        # Find PA(s) that contain the clicked point
        matched = pan_india_pa.filterBounds(point)
        count   = matched.size().getInfo()

        if count == 0:
            info_html.value = """
            <div style='padding:12px 16px;font-family:sans-serif;font-size:13px;
                        background:white;border:1px solid #ccc;border-radius:6px;
                        box-shadow:2px 2px 6px rgba(0,0,0,0.15);'>
                <span style='color:#888;'>No protected area at this location.<br>
                Try clicking closer to a boundary.</span>
            </div>"""
            return

        # Take the first matching feature
        feature   = ee.Feature(matched.first())
        props     = feature.toDictionary().getInfo()
        info_html.value = build_info_html(props)

    except Exception as e:
        info_html.value = f"""
        <div style='padding:12px 16px;font-family:sans-serif;font-size:13px;
                    background:white;border:1px solid #ccc;border-radius:6px;'>
            <span style='color:red;'>Error: {e}</span>
        </div>"""

Map3.on_interaction(handle_click)

# ── Display map + info panel side by side ─────────────────────────────────────
dashboard = widgets.HBox(
    [Map3, info_html],
    layout=widgets.Layout(align_items='flex-start', gap='12px')
)
dashboard


✓ Loaded Pan India Protected Areas
✓ Loaded Pan India Terrain Raster


## 9. Summary Statistics

Compute area statistics for protected areas in this tehsil.

In [10]:
if pa_count > 0:
    # Define tree cover change classes
    class_values = [-2, -1, 0, 1, 2, 3, 4, 5]

    # Get pixel area in hectares
    pixel_area_ha = ee.Image.pixelArea().divide(10000)

    # Calculate areas for each class within protected areas
    stats_list = []

    for pa_feature in protected_areas_filtered.toList(pa_count).getInfo():
        pa_name = pa_feature['properties'].get('name', 'Unknown')
        pa_geom_single = ee.Geometry(pa_feature['geometry'])

        # Clip tree change to this PA
        tree_change_single_pa = tree_change.clip(pa_geom_single)

        # Calculate area for each class
        class_areas = {}
        for class_val in class_values:
            area = pixel_area_ha.updateMask(tree_change_single_pa.eq(class_val)) \
                .reduceRegion(
                    reducer=ee.Reducer.sum(),
                    geometry=pa_geom_single,
                    scale=30,
                    maxPixels=1e13
                ).get('area')

            area_val = area.getInfo()
            class_areas[CHANGE_LABELS[class_val]] = round(area_val if area_val else 0, 2)

        stats_list.append({
            'Protected Area': pa_name,
            **class_areas
        })

    # Display as table
    import pandas as pd
    df_stats = pd.DataFrame(stats_list)

    print("\n" + "="*80)
    print(f"Tree Cover Change Statistics (Area in Hectares)")
    print(f"Location: {STATE.title()} > {DISTRICT.title()} > {TEHSIL.title()}")
    print("="*80 + "\n")

    display(df_stats)

else:
    print("\nNo protected areas found in this tehsil for statistics calculation.")


Tree Cover Change Statistics (Area in Hectares)
Location: Odisha > Anugul > Anugul



,Protected Area,Tree Cover Loss,Degradation,No Change,Improvement,Tree Cover Gain,Partially Degraded,Missing Data
0,Satkosia Gorge WLS,2051.66,16394.46,24479.31,4933.58,1216.63,375.31,1036.89


## 10. Export Data (Optional)

Export the filtered protected areas with statistics to Google Drive or as a GEE asset.

In [ ]:
# Uncomment to export statistics to CSV
# if pa_count > 0:
#     export_filename = f'protected_areas_stats_{STATE}_{DISTRICT}_{TEHSIL}.csv'
#     df_stats.to_csv(export_filename, index=False)
#     print(f"✓ Statistics exported to {export_filename}")

# Uncomment to export to Google Drive (requires GEE task)
# if pa_count > 0:
#     task = ee.batch.Export.table.toDrive(
#         collection=protected_areas_filtered,
#         description=f'PA_Export_{STATE}_{DISTRICT}_{TEHSIL}',
#         fileFormat='GeoJSON'
#     )
#     task.start()
#     print(f"✓ Export task started: {task.id}")

print("Export options available (uncomment code above to use)")

---

## References

**Data Sources:**
- [Tree Cover Change Maps](https://www.cse.iitd.ernet.in/~aseth/forest-health-ictd2024.pdf)
- [Protected Area Boundaries Database](https://pau-database.kalpavriksh.org/)
- CoRE Stack GEE Platform

**Project:** CoRE Stack Innovation Challenge Demo (February 2026)

**Analysis Period:** 2017-2023

---